# 스미싱 URL 탐색기 · CANINE 학습 (Colab)

1. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 선택
2. 내 드라이브에 `smishing/data/` 폴더를 만들고 `train.csv`, `val.csv`, `test.csv`, `test_phishtank_recent.csv`, `train_canine.py`를 올린다
3. 왼쪽 🔑(보안 비밀)에 `HF_TOKEN`을 넣는다 (모델을 Hugging Face에 올릴 때만 필요)
4. 위에서부터 차례로 실행. 끊기면 2·4번 셀만 다시 실행하면 이어서 학습한다.

In [ ]:
!pip -q install transformers==4.47.1 accelerate scikit-learn pandas
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/smishing'
!ls -la $D/data

In [ ]:
# 빠른 시험: 2천 건으로 몇 분만 돌려서 오류가 없는지 확인
!python $D/data/train_canine.py --data $D/data --out $D/out_test --epochs 1 --limit 2000

In [ ]:
# 본 학습 (끊기면 이 셀만 다시 실행 → 이어서 학습)
import os
from google.colab import userdata
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass
PUSH = ''  # 예: '팀아이디/smishing-canine' (비우면 드라이브에만 저장)
!python $D/data/train_canine.py --data $D/data --out $D/out --epochs 3 {'--push ' + PUSH if PUSH else ''}

In [ ]:
import json
print(json.dumps(json.load(open(f'{D}/out/metrics.json')), indent=2, ensure_ascii=False))

## v2: 2025~2026년 PhishTank까지 넣어 다시 학습
1. 내 드라이브 `smishing/` 안에 `data_v2/` 폴더를 만들고, 노트북 `training/colab_upload_v2/`의 파일 4개(`train.csv`, `val.csv`, `test.csv`, `train_canine.py`)를 올린다
2. 맨 위 1·2번 셀(설치, 드라이브 연결)을 실행한 뒤 아래 셀을 실행한다. 끊기면 2번 셀과 아래 셀만 다시 실행하면 이어서 학습한다.
3. 나누기: 학습 ~2025-12 / 검증 2026-01~04 / 테스트 2026-05~10

In [ ]:
!python $D/data_v2/train_canine.py --data $D/data_v2 --out $D/out_v2 --epochs 3

In [ ]:
import json
print(json.dumps(json.load(open(f'{D}/out_v2/metrics.json')), indent=2, ensure_ascii=False))